## Adaptive RAG — QPP-guided pipeline with λ(q) context-influence weighting

Each query q receives a two-component adaptive decision:
- **k(q)** : how many passages to retrieve
- **λ(q)** : how strongly those passages influence the generated answer

Both come from the QPP score s via:
```
k(q)  = k_min + round((k_max - k_min) * s)
λ(q)  = 1 / (1 + exp(-6 * (s - 0.5)))
```

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.evaluate import compute_generation_metrics

sns.set_theme(style='whitegrid')

### λ(q) implementation

High QPP score → λ close to 1 → generator instructed to rely on retrieved passages.  
Low QPP score → λ close to 0 → generator falls back on its own parametric knowledge.

In [ ]:
def compute_lambda(qpp_score, alpha=6.0):
    """sigmoid centred at 0.5 -- maps QPP score to context-influence weight lambda"""
    return 1.0 / (1.0 + math.exp(-alpha * (qpp_score - 0.5)))

def compute_k(qpp_score, k_min=5, k_max=20):
    """linear map from QPP score to retrieval depth"""
    return k_min + round((k_max - k_min) * qpp_score)

def make_prompt(query, context, lambda_q):
    """
    Build generation prompt conditioned on lambda_q.
    This is where lambda(q) concretely affects generation --
    the instruction to the generator changes based on estimated retrieval quality.

      lambda_q > 0.7  -> passages are the primary source
      lambda_q > 0.4  -> balanced use of passages and model knowledge
      lambda_q <= 0.4 -> model answers from its own knowledge; passages are background only
    """
    if lambda_q > 0.7:
        prompt = f"Using the following passages as your primary source:\n{context}\n\nAnswer: {query}"
    elif lambda_q > 0.4:
        prompt = f"Consider these passages, but also use your own knowledge:\n{context}\n\nAnswer: {query}"
    else:
        prompt = f"Answer from your own knowledge. Additional context (use sparingly):\n{context}\n\nAnswer: {query}"
    return prompt

def lambda_to_regime(lam):
    if lam > 0.7:
        return 'High-λ'
    elif lam > 0.4:
        return 'Mid-λ'
    return 'Low-λ'

### Load results and attach λ values

Results saved by the updated pipeline already contain `lambda_used`. If running on older results, we recompute from `qpp_score`.

In [ ]:
baseline_rows = [json.loads(l) for l in open('../outputs/predictions/baseline_results.jsonl')]
qpp_rows      = [json.loads(l) for l in open('../outputs/predictions/adaptive_rag_results.jsonl')]

df_base = pd.DataFrame(baseline_rows)
df_qpp  = pd.DataFrame(qpp_rows)

# compute lambda if not already in the saved results
if 'lambda_used' not in df_qpp.columns:
    df_qpp['lambda_used'] = df_qpp['qpp_score'].apply(compute_lambda)

df_qpp['lambda_regime'] = df_qpp['lambda_used'].apply(lambda_to_regime)

print(f'Baseline:    {len(df_base)} queries')
print(f'QPP-guided:  {len(df_qpp)} queries')
print()
print(df_qpp[['qpp_score', 'k_used', 'lambda_used', 'rouge_l', 'bert_f1', 'token_f1']].describe().round(4))

### λ distribution across queries

In [ ]:
print('Lambda regime breakdown:')
print(df_qpp['lambda_regime'].value_counts())
print()
print(f"Mean λ : {df_qpp['lambda_used'].mean():.4f}")
print(f"Std  λ : {df_qpp['lambda_used'].std():.4f}")

### Overall comparison: Baseline vs QPP-guided

In [ ]:
df_compare = pd.DataFrame({
    'Method':    ['Baseline RAG', 'QPP-Guided RAG'],
    'ROUGE-L':   [df_base['rouge_l'].mean(),  df_qpp['rouge_l'].mean()],
    'BERTScore': [df_base['bert_f1'].mean(),  df_qpp['bert_f1'].mean()],
    'F1':        [df_base['token_f1'].mean(), df_qpp['token_f1'].mean()],
}).set_index('Method')

df_compare.loc['Gain'] = df_compare.loc['QPP-Guided RAG'] - df_compare.loc['Baseline RAG']
df_compare.to_csv('../outputs/predictions/baseline_vs_qpp.csv')
print(df_compare.round(4))

### Gain breakdown by QPP group — with mean λ per group

In [ ]:
qpp_scores = df_qpp['qpp_score'].values.astype(float)
t1, t2 = np.percentile(qpp_scores, 33), np.percentile(qpp_scores, 66)

groups = {
    'Low-QPP':    qpp_scores < t1,
    'Medium-QPP': (qpp_scores >= t1) & (qpp_scores < t2),
    'High-QPP':   qpp_scores >= t2,
}

records = []
for label, mask in groups.items():
    b   = df_base['rouge_l'].values[mask].mean()
    g   = df_qpp['rouge_l'].values[mask].mean()
    lam = df_qpp['lambda_used'].values[mask].mean()   # mean lambda in this group
    records.append({
        'QPP Group':  label,
        'N':          int(mask.sum()),
        'Mean λ':     round(lam, 4),
        'Baseline':   round(b, 4),
        'QPP-Guided': round(g, 4),
        'Gain':       round(g - b, 4),
    })

df_groups = pd.DataFrame(records).set_index('QPP Group')
df_groups.to_csv('../outputs/predictions/gains_by_group.csv')
print(df_groups)

### Retrieval depth and λ distributions

In [ ]:
k_counts = df_qpp['k_used'].value_counts().sort_index()
print('Retrieval depth (k) distribution:')
print(k_counts)
print()
print('Lambda regime counts:')
print(df_qpp['lambda_regime'].value_counts())

### Visualisations

Four panels:
1. Per-query ROUGE-L: baseline vs QPP-guided
2. ROUGE-L gain by QPP group
3. λ(q) histogram with threshold lines
4. λ(q) vs per-query ROUGE-L gain

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

base_rouge  = df_base['rouge_l'].values
qpp_rouge   = df_qpp['rouge_l'].values
lambda_vals = df_qpp['lambda_used'].values

scatter_colors = np.where(qpp_scores < t1, '#E53935',
                 np.where(qpp_scores < t2, '#FB8C00', '#43A047'))

# panel 1: scatter baseline vs QPP-guided
axes[0, 0].scatter(base_rouge, qpp_rouge, c=scatter_colors, alpha=0.5, s=15)
lim = [0.0, max(base_rouge.max(), qpp_rouge.max()) + 0.05]
axes[0, 0].plot(lim, lim, 'k--', lw=1, alpha=0.5, label='No change')
axes[0, 0].set_xlabel('Baseline ROUGE-L')
axes[0, 0].set_ylabel('QPP-Guided ROUGE-L')
axes[0, 0].set_title('Per-Query ROUGE-L Improvement')
patches = [
    mpatches.Patch(color='#E53935', label='Low-QPP  (k=50)'),
    mpatches.Patch(color='#FB8C00', label='Medium-QPP (k=30)'),
    mpatches.Patch(color='#43A047', label='High-QPP  (k=20)'),
]
axes[0, 0].legend(handles=patches, fontsize=9)

# panel 2: bar chart gains by QPP group
group_labels  = df_groups.index.tolist()
baseline_vals = df_groups['Baseline'].values
qpp_vals      = df_groups['QPP-Guided'].values
x = np.arange(len(group_labels))
w = 0.35
axes[0, 1].bar(x - w/2, baseline_vals, w, label='Baseline RAG',   color='#90A4AE')
axes[0, 1].bar(x + w/2, qpp_vals,      w, label='QPP-Guided RAG', color='#1E88E5')
for i, (b, g) in enumerate(zip(baseline_vals, qpp_vals)):
    axes[0, 1].text(i + w/2, g + 0.003, f'+{g-b:.3f}', ha='center',
                    fontsize=9, color='#1E88E5', fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(group_labels)
axes[0, 1].set_ylabel('ROUGE-L')
axes[0, 1].set_title('ROUGE-L Gain by QPP Group')
axes[0, 1].legend()

# panel 3: lambda histogram
axes[1, 0].hist(lambda_vals, bins=30, color='#7986CB', edgecolor='white')
axes[1, 0].axvline(0.4, color='#FB8C00', linestyle='--', lw=1.2, label='λ=0.4 (low/mid cutoff)')
axes[1, 0].axvline(0.7, color='#43A047', linestyle='--', lw=1.2, label='λ=0.7 (mid/high cutoff)')
axes[1, 0].set_xlabel('λ(q) — context-influence weight')
axes[1, 0].set_ylabel('Number of queries')
axes[1, 0].set_title('Distribution of λ(q) across queries')
axes[1, 0].legend(fontsize=9)

# panel 4: lambda vs ROUGE-L gain
rouge_gain = qpp_rouge - base_rouge
axes[1, 1].scatter(lambda_vals, rouge_gain, c=scatter_colors, alpha=0.4, s=15)
axes[1, 1].axhline(0, color='black', linestyle='--', lw=0.8, alpha=0.5)
axes[1, 1].axvline(0.4, color='#FB8C00', linestyle='--', lw=1.0, alpha=0.6)
axes[1, 1].axvline(0.7, color='#43A047', linestyle='--', lw=1.0, alpha=0.6)
axes[1, 1].set_xlabel('λ(q) — context-influence weight')
axes[1, 1].set_ylabel('ROUGE-L gain (QPP-guided − Baseline)')
axes[1, 1].set_title('λ(q) vs Per-Query ROUGE-L Gain')
patches2 = [
    mpatches.Patch(color='#E53935', label='Low-QPP'),
    mpatches.Patch(color='#FB8C00', label='Medium-QPP'),
    mpatches.Patch(color='#43A047', label='High-QPP'),
]
axes[1, 1].legend(handles=patches2, fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/adaptive_rag_gains.png', dpi=150, bbox_inches='tight')
plt.show()

### Metrics broken down by λ regime

Confirms whether the prompt-conditioning change caused by λ is having the expected effect on generation quality.

In [ ]:
regime_order = ['Low-λ', 'Mid-λ', 'High-λ']

regime_summary = (
    df_qpp.groupby('lambda_regime')[['rouge_l', 'bert_f1', 'token_f1', 'lambda_used']]
    .mean()
    .round(4)
    .reindex([r for r in regime_order if r in df_qpp['lambda_regime'].unique()])
)
regime_summary.columns = ['ROUGE-L', 'BERTScore-F1', 'Token-F1', 'Mean λ']
regime_summary.to_csv('../outputs/predictions/results_by_lambda_regime.csv')
print('Mean metrics per λ regime:')
print(regime_summary)

### Prompt examples for each λ regime

Shows the reviewer exactly how the prompt instruction changes with λ(q).

In [ ]:
example_query   = "What causes the northern lights?"
example_context = "[Retrieved passage text would appear here]"

for lam_val in [0.15, 0.55, 0.85]:
    print(f"--- λ = {lam_val} ({lambda_to_regime(lam_val)}) ---")
    print(make_prompt(example_query, example_context, lam_val))
    print()